# California Housing — Model Development and Evaluation

        ## Objective

        Train one reproducible random-forest baseline for the historical California Housing target. The goal is a transparent benchmark with an untouched test set—not a production valuation system.

        **Evaluation promise:** split first; fit every learned transformation on training rows only; generate one prediction for each holdout row; interpret several complementary metrics together.


## Why this pipeline

        ```text
        source rows → fixed split → [fit imputer → fit scaler → fit forest] on train
                    → transform/predict test with training state → metrics and diagnostics
        ```

        A `Pipeline` makes this order executable. It prevents a later edit from accidentally fitting preprocessing statistics on the holdout set.


In [ ]:
# Use the maintained loader and the small, test-covered project helpers.
        # The repository root must be the current directory when running this notebook;
        # use `jupyter nbconvert --execute notebooks/02_model_development.ipynb`
        # from the project root for a reproducible command-line run.
        from sklearn.datasets import fetch_california_housing
        import matplotlib.pyplot as plt
        import pandas as pd
        import seaborn as sns

        from src.housing_model import build_pipeline, evaluate_predictions, feature_importance, split_data

        sns.set_theme(style="whitegrid", context="notebook")
        RANDOM_STATE = 42


In [ ]:
# Fetch named columns and preserve the target's unit ($100,000s).
        # We do not commit an opaque data copy; scikit-learn caches the public source.
        housing = fetch_california_housing(as_frame=True)
        X = housing.data
        y = housing.target
        print(f"Source rows: {len(X):,} | Feature columns: {list(X.columns)}")


## 1. Reserve the holdout before fitting anything

        The test set is a simulation of future unseen districts. It cannot influence imputation medians, scaling parameters, tree splits, hyperparameter choice, or final reported metrics. `random_state=42` makes this baseline repeatable, but does **not** prove geographic generalisation.


In [ ]:
# Split raw rows first. The helper is unit tested for reproducibility and
        # conservation of all rows, which protects this critical evaluation boundary.
        X_train, X_test, y_train, y_test = split_data(
            X, y, test_size=0.20, random_state=RANDOM_STATE
        )
        print(f"Training rows: {len(X_train):,}")
        print(f"Untouched test rows: {len(X_test):,}")


## 2. Fit the baseline

        The pipeline uses median imputation, standard scaling, and a 300-tree random forest with at least two rows per leaf. Trees do not require scaling; the scaler is retained inside the pipeline so the preprocessing contract is reusable for a future linear or distance-based comparison. It is not the reason the forest works.


In [ ]:
# `.fit` is called only on training data. The pipeline forwards each stage
        # in order and stores the learned state needed later to transform test rows.
        pipeline = build_pipeline(random_state=RANDOM_STATE)
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)

        # A shape check is a compact guard against accidental row loss or broadcasting.
        assert predictions.shape == (len(X_test),)


## 3. Measure complementary error views

        - **MAE**: average absolute miss in target units; easiest to translate to dollars.
        - **RMSE**: penalises larger misses more heavily, so it is sensitive to tails.
        - **R²**: fraction of holdout target variance captured relative to a constant baseline.

        None of these says that every geography or price range has the same error.


In [ ]:
# Keep evaluation in the target's native unit, then add a dollar translation
        # only as an aid to interpretation. The conversion does not change the metric.
        metrics = evaluate_predictions(y_test, predictions)
        metric_table = pd.DataFrame([metrics], index=["Random forest baseline"])
        display(metric_table.round(3))
        print(f"Typical MAE in dollars: approximately ${metrics['mae'] * 100_000:,.0f}")


In [ ]:
# Diagnostics reveal patterns a scalar score hides. The diagonal is perfect
        # prediction; residuals above zero mean the model predicted too low.
        residuals = y_test - predictions
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.7))
        axes[0].scatter(y_test, predictions, alpha=0.28, s=12, color="#247BA0")
        limits = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
        axes[0].plot(limits, limits, "--", color="#E45756", label="perfect prediction")
        axes[0].set(title="Holdout predictions versus actual", xlabel="Actual ($100,000s)", ylabel="Predicted ($100,000s)")
        axes[0].legend()
        sns.histplot(residuals, bins=38, kde=True, color="#5B3FD6", ax=axes[1])
        axes[1].axvline(0, color="#152033", linestyle="--")
        axes[1].set(title="Holdout residual distribution", xlabel="Actual − predicted ($100,000s)")
        fig.tight_layout()
        plt.show()


## 4. Inspect feature reliance without claiming cause

        Random-forest impurity importance describes how useful a feature was for splits in this fitted model. It is affected by correlated features and split opportunities. It does **not** mean changing median income, latitude, or any other column would mechanically change house value.


In [ ]:
# The helper sorts the fitted forest's importances descending and is tested
        # independently. Plot all eight values to avoid hiding lower-ranked features.
        importance = pd.Series(feature_importance(pipeline, list(X.columns)))
        fig, ax = plt.subplots(figsize=(9, 4.8))
        importance.sort_values().plot.barh(ax=ax, color="#5B3FD6")
        ax.set(title="Impurity-based feature reliance", xlabel="Importance")
        plt.tight_layout()
        plt.show()


## 5. Decision record and next experiment

        This baseline supports a reproducible 1990 district-level benchmark. It does not support real-estate appraisal, current pricing, causal interpretation, or geographic-transfer claims.

        The most valuable next experiment is **spatially blocked cross-validation**: form geographic groups first, train on some regions, and test on withheld regions. Pair that with permutation importance and error slices by location and target range before trusting any deployment-oriented conclusion.
